# Description

Predicts drug-disease associations using the **module-based** approach: projections of SMulTiXcan (disease z-scores) and LINCS L1000 (drug z-scores) into the ARCHS4 CLAMP latent space.

The prediction score for a drug-disease pair is:
$$\text{score} = -1 \times \mathbf{drug}^T \mathbf{disease}$$

A higher score means that the drug reverses the transcriptomic signature of the disease (sign reversal across LVs), suggesting therapeutic potential.

This follows the framework of [Menden et al., 2020 (Nature Neuroscience)](https://doi.org/10.1038/nn.4618).

Results are saved as HDF5 files with keys:
- `full_prediction`: predictions for all traits
- `prediction`: predictions mapped to DOID (for comparison with PharmacotherapyDB gold standard)
- `metadata`: method name, n_top_lvs, data source

# Module loading

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [18]:
# if True, re-run even if output files already exist
FORCE_RUN = True

PREDICTION_METHOD = 'Module-based'

In [19]:
DATA_DIR = here('data/archs4/drug_diseases_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

PROJECTIONS_DIR = here('output/drug_disease_analyses')
display(PROJECTIONS_DIR)
assert PROJECTIONS_DIR.exists()

OUTPUT_PREDICTIONS_DIR = PROJECTIONS_DIR / 'predictions' / 'dotprod_neg'
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_PREDICTIONS_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg')

# Helper functions

In [20]:
def map_traits_to_doid(data, preferred_doids, ukb_efo, efo_xrefs, do_xrefs):
    """
    Maps trait columns (UKB full codes) to Disease Ontology IDs (DOID).

    For traits mapping to multiple DOIDs, prefers those in `preferred_doids`
    (DOIDs present in the gold standard). When a DOID appears from multiple
    traits, keeps the maximum score.

    Args:
        data: DataFrame with traits as columns (UKB full-code format).
        preferred_doids: set of DOID strings present in the gold standard.
        ukb_efo: DataFrame indexed by ukb_fullcode, with column 'term_codes'
                 (comma-separated EFO codes).
        efo_xrefs: DataFrame with columns ['term_id', 'target_id_type', 'target_id']
                   (maps EFO IDs to DOID via exact EFO:XXXXXXX format).
        do_xrefs: DataFrame with columns ['doid_code', 'resource', 'resource_id']
                  (maps EFO numbers without 'EFO:' prefix to DOID codes).

    Returns:
        DataFrame with DOID columns. Traits without a DOID mapping are dropped.
        When multiple traits map to the same DOID, maximum score is kept.
    """
    doid_efo = efo_xrefs[efo_xrefs['target_id_type'] == 'DOID']

    trait_to_doid = {}
    for trait in data.columns:
        if trait not in ukb_efo.index:
            continue
        rows = ukb_efo.loc[trait]
        if isinstance(rows, pd.Series):
            rows = rows.to_frame().T

        all_efo_codes = set()
        for term_codes in rows['term_codes'].dropna():
            for code in str(term_codes).split(','):
                code = code.strip()
                if code:
                    all_efo_codes.add(code)

        all_doids = set()
        for efo_code in all_efo_codes:
            mask = doid_efo['term_id'] == efo_code
            all_doids.update(doid_efo[mask]['target_id'].values)
            if efo_code.startswith('EFO:'):
                efo_num = efo_code[4:]
                mask2 = (do_xrefs['resource'] == 'EFO') & (do_xrefs['resource_id'] == efo_num)
                all_doids.update(do_xrefs[mask2]['doid_code'].values)

        if not all_doids:
            continue

        preferred = sorted(all_doids & preferred_doids)
        trait_to_doid[trait] = preferred[0] if preferred else sorted(all_doids)[0]

    data_mapped = data.loc[:, list(trait_to_doid.keys())].rename(columns=trait_to_doid)
    data_mapped = data_mapped.T.groupby(level=0).max().T
    return data_mapped

In [21]:
def zero_nontop_lvs(trait_vector, n_top, use_abs=True):
    """Zeros all but the top `n_top` LV values in a Series."""
    values = trait_vector.abs() if use_abs else trait_vector
    top_idx = values.sort_values(ascending=False).head(n_top).index
    result = trait_vector.copy()
    result[~result.index.isin(top_idx)] = 0.0
    return result

In [22]:
def predict_and_save(
    lincs_proj,
    smultixcan_proj,
    output_dir,
    doids_in_gold_standard,
    trait_to_doid_func,
    method_name,
    data_stem,
    n_top=None,
    use_abs=True,
    force_run=True,
):
    """
    Computes dot-product drug-disease predictions and saves to HDF5.
    score = -1 * drug^T * disease
    """
    suffix = 'all_genes' if n_top is None else f'top_{n_top}_genes'
    output_file = output_dir / f'{data_stem}-{suffix}-prediction_scores.h5'

    print(f'predicting {suffix}...', end='')
    if output_file.exists() and not force_run:
        print('  already run')
        return
    print('')

    disease_data = smultixcan_proj.copy()
    if n_top is not None:
        disease_data = disease_data.apply(lambda x: zero_nontop_lvs(x, n_top, use_abs))

    # score = -1 * (LVs x drugs)^T dot (LVs x traits) → drugs x traits
    scores = -1.0 * lincs_proj.T.dot(disease_data)
    print(f'  shape: {scores.shape}')

    with pd.HDFStore(output_file, mode='w', complevel=4) as store:
        # full prediction
        scores.index.name = 'drug'
        scores.columns.name = 'trait'
        full_pred = (
            scores.unstack()
            .reset_index()
            .rename(columns={0: 'score'})
        )
        full_pred['trait'] = full_pred['trait'].astype('category')
        full_pred['drug'] = full_pred['drug'].astype('category')
        assert full_pred.shape == full_pred.dropna().shape
        print(f'  full_prediction shape: {full_pred.shape}')
        display(full_pred.describe())
        store.put('full_prediction', full_pred, format='table')

        # DOID-mapped prediction
        scores_doid = trait_to_doid_func(scores)
        print(f'  shape after DOID map: {scores_doid.shape}')
        assert scores_doid.index.is_unique
        assert scores_doid.columns.is_unique

        scores_doid.index.name = 'drug'
        scores_doid.columns.name = 'trait'
        doid_pred = (
            scores_doid.unstack()
            .reset_index()
            .rename(columns={0: 'score'})
        )
        doid_pred['trait'] = doid_pred['trait'].astype('category')
        doid_pred['drug'] = doid_pred['drug'].astype('category')
        assert doid_pred.shape == doid_pred.dropna().shape
        print(f'  prediction shape: {doid_pred.shape}')
        store.put('prediction', doid_pred, format='table')

        meta = pd.DataFrame({
            'method': [method_name],
            'n_top_genes': [-1.0 if n_top is None else float(n_top)],
            'data': [data_stem],
        })
        store.put('metadata', meta, format='table')

    print(f'  saved to: {output_file}')

# Load PharmacotherapyDB gold standard

In [23]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

doids_in_gold_standard = set(gold_standard['trait'])
print(f'Unique DOIDs in gold standard: {len(doids_in_gold_standard)}')

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


Unique DOIDs in gold standard: 87


# Load trait→DOID mapping files

In [24]:
ukb_efo = pd.read_csv(
    DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv',
    sep='\t',
    index_col='ukb_fullcode',
)
display(ukb_efo.shape)
display(ukb_efo.head())

(1087, 6)

,ukb_code,term_label,term_codes,mapping_type,current_term_label,category
ukb_fullcode,,,,,,
K55-Diagnoses_main_ICD10_K55_Vascular_disorders_of_intestine,K55,vascular disease,"EFO:0004264, EFO:0009431",Broad,vascular disease AND intestinal disease,disease
M17-Diagnoses_main_ICD10_M17_Gonarthrosis_arthrosis_of_knee,M17,osteoarthritis || knee,EFO:0004616,Broad,"osteoarthritis, knee",disease
R30-Diagnoses_main_ICD10_R30_Pain_associated_with_micturition,R30,dysuria,EFO:0003901,? Broad,dysuria,NaN
O60-Diagnoses_main_ICD10_O60_Preterm_delivery,O60,premature birth,EFO:0003917,? Exact,premature birth,NaN
S64-Diagnoses_main_ICD10_S64_Injury_of_nerves_at_wrist_and_hand_level,S64,carpal tunnel syndrome,EFO:0004143,? Narrow,carpal tunnel syndrome,disease


In [25]:
efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
display(efo_xrefs.shape)
display(efo_xrefs.head())

(104094, 3)

,term_id,target_id_type,target_id
0,Orphanet:284232,DOID,DOID:0110175
1,Orphanet:284232,GARD,GARD:0012434
2,Orphanet:284232,ICD10,ICD10:G60.0
3,Orphanet:284232,MONDO,MONDO:0013644
4,Orphanet:284232,OMIM,OMIM:614228


In [26]:
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')
display(do_xrefs.shape)
display(do_xrefs.head())

(8204, 4)

,doid_code,doid_name,resource,resource_id
0,DOID:2531,hematologic cancer,CSP,2004-1600
1,DOID:2531,hematologic cancer,CSP,2004-1803
2,DOID:2531,hematologic cancer,CSP,2004-2820
3,DOID:2531,hematologic cancer,EFO,0000095
4,DOID:2531,hematologic cancer,EFO,0000096


In [27]:
def trait_to_doid_func(data):
    return map_traits_to_doid(data, doids_in_gold_standard, ukb_efo, efo_xrefs, do_xrefs)

# Load projected data

In [28]:
lincs_proj = pd.read_pickle(PROJECTIONS_DIR / 'lincs-projection.pkl')
print(f'LINCS projection shape: {lincs_proj.shape}')
display(lincs_proj.head())

LINCS projection shape: (2366, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,-0.003664,0.021802,0.050683,-0.023411,0.054818,0.022504,-0.016074,-0.007949,-0.012020,-0.022674,...,-0.051500,-0.039418,-0.021987,0.012250,0.009535,0.012377,0.013959,-0.209899,0.026002,0.009327
LV2,0.011797,0.058622,0.001288,-0.023493,0.009715,0.001838,-0.025821,-0.026784,-0.010831,0.000237,...,-0.003300,-0.000286,0.004965,0.011235,0.002291,-0.003366,-0.002029,-0.067440,0.005340,0.011255
LV3,-0.000841,-0.055107,-0.006764,0.027218,-0.012003,-0.010472,-0.021370,-0.007238,-0.005116,0.005954,...,-0.003723,0.009973,-0.007469,0.009228,-0.010771,-0.017754,0.012783,0.066089,0.008973,-0.011303
LV4,0.031992,-0.511423,-0.071332,-0.059371,-0.035587,-0.033645,-0.076674,0.042806,-0.099315,0.031507,...,0.051060,0.022656,0.054687,0.014682,-0.078138,-0.014696,0.015666,-0.238956,-0.018649,-0.047013
LV5,-0.032510,0.123403,0.005948,0.031886,-0.045581,-0.006246,0.092339,0.034634,0.111000,-0.025291,...,-0.020998,-0.019916,-0.030282,0.009627,0.031024,-0.017982,-0.019684,0.164316,0.012215,-0.027974


In [29]:
smultixcan_proj = pd.read_pickle(PROJECTIONS_DIR / 'smultixcan-mashr-zscores-projection.pkl')
print(f'SMulTiXcan projection shape: {smultixcan_proj.shape}')
display(smultixcan_proj.head())

SMulTiXcan projection shape: (2366, 4091)


,20096_1-Size_of_red_wine_glass_drunk_small_125ml,2345-Ever_had_bowel_cancer_screening,N49-Diagnoses_main_ICD10_N49_Inflammatory_disorders_of_male_genital_organs_not_elsewhere_classified,100011_raw-Iron,5221-Index_of_best_refractometry_result_right,20003_1141150624-Treatmentmedication_code_zomig_25mg_tablet,S69-Diagnoses_main_ICD10_S69_Other_and_unspecified_injuries_of_wrist_and_hand,20024_1136-Job_code_deduced_Information_and_communication_technology_managers,20002_1385-Noncancer_illness_code_selfreported_allergy_or_anaphylactic_reaction_to_food,G6_SLEEPAPNO-Sleep_apnoea,...,Astle_et_al_2016_Sum_basophil_neutrophil_counts,RA_OKADA_TRANS_ETHNIC,pgc.scz2,PGC_ADHD_EUR_2017,MAGIC_FastingGlucose,Astle_et_al_2016_Red_blood_cell_count,SSGAC_Depressive_Symptoms,BCAC_ER_positive_BreastCancer_EUR,IBD.EUR.Inflammatory_Bowel_Disease,Astle_et_al_2016_High_light_scatter_reticulocyte_count
LV1,0.153456,0.171552,0.153334,0.152908,0.134566,0.163124,0.145335,0.146773,0.152961,0.142850,...,0.262381,0.196087,0.286988,0.163989,0.139514,0.285309,0.157414,0.156926,0.194079,0.297641
LV2,0.100974,0.106346,0.101445,0.095510,0.088895,0.089579,0.089118,0.090771,0.082718,0.102523,...,0.172808,0.141204,0.143710,0.121239,0.090788,0.199298,0.095521,0.101541,0.142240,0.164590
LV3,0.070661,0.083573,0.082994,0.065916,0.073920,0.086248,0.073543,0.076102,0.061243,0.069460,...,0.073540,0.090083,0.152892,0.113261,0.060385,0.090092,0.089826,0.072068,0.070305,0.086717
LV4,0.103140,0.111902,0.096378,0.085740,0.107958,0.111800,0.124310,0.087583,0.109811,0.118070,...,0.136074,0.113084,0.168764,0.136986,0.092374,0.133274,0.133228,0.112045,0.119645,0.094503
LV5,0.035550,0.015937,0.014084,-0.014893,0.015324,0.027896,0.024724,0.020440,0.018119,0.027276,...,0.042351,0.049341,0.024278,0.017188,0.025756,0.071969,0.003834,0.013244,0.049691,0.020399


# Predict drug-disease associations

In [30]:
# Top-N LV thresholds (None = use all LVs)
N_TOP_LVS_LIST = [None, 5, 10, 25, 50]

DATA_STEM = 'smultixcan-mashr-zscores-projection'

for n_top in N_TOP_LVS_LIST:
    predict_and_save(
        lincs_proj=lincs_proj,
        smultixcan_proj=smultixcan_proj,
        output_dir=OUTPUT_PREDICTIONS_DIR,
        doids_in_gold_standard=doids_in_gold_standard,
        trait_to_doid_func=trait_to_doid_func,
        method_name=PREDICTION_METHOD,
        data_stem=DATA_STEM,
        n_top=n_top,
        use_abs=True,
        force_run=FORCE_RUN,
    )
    print()

predicting all_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,3.414745e-03
std,6.212713e-02
min,-6.076442e+00
25%,-1.591993e-02
50%,-8.343580e-04
75%,1.550039e-02
max,3.340060e+00


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-projection-all_genes-prediction_scores.h5

predicting top_5_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,7.417630e-03
std,3.799051e-02
min,-4.084294e+00
25%,-6.946079e-03
50%,1.159469e-03
75%,1.148076e-02
max,1.488175e+00


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-projection-top_5_genes-prediction_scores.h5

predicting top_10_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


...

  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-projection-top_10_genes-prediction_scores.h5

predicting top_25_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,7.386291e-03
std,4.776381e-02
min,-8.417966e+00
25%,-9.223926e-03
50%,1.125430e-03
75%,1.440140e-02
max,2.517419e+00


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-projection-top_25_genes-prediction_scores.h5

predicting top_50_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,7.211886e-03
std,5.391556e-02
min,-5.967197e+00
25%,-1.041384e-02
50%,8.414758e-04
75%,1.497938e-02
max,3.069565e+00


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-projection-top_50_genes-prediction_scores.h5

